# Get to Know a Dataset: 
# NeMO Human and Mammalian Brain Atlas BRAIN Consortium Data

The Neuroscience Multi-Omic Archive (NeMO Archive; (nemoarchive.org)[nemoarchive.org] serves as the primary repository for genomics data from the [Brain Initiative](https://braininitiative.nih.gov/). Our data is a curated resource containing transcriptomic and epigenomic data from over 50 million brain cells, including single-cell genomic data from major regions of adult and prenatal human and mouse brains, as well as substantial single-cell genomic data from non-human primates.

This notebook serves as a guided tour of the NeMO Human and Mammalian Brain Atlas (https://registry.opendata.aws/[NeMO_archive.yaml]) dataset. More usage examples, tutorials for other AWS Open Data projects can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

Please have fun and thank you for exploring the NeMO Human and Mammalian Brain Atlas BRAIN Consortium Data!

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

At the top level of our S3 bucket, we have a single key prefix "nemo-public-preview.s3" that in turn contains:

 1. README.txt
 2. License.txt
 2. data/
 3. metadata/
 4. docs/

Additional information can be found here: 
- Documentation for the NeMO archive: https://github.com/nemoarchive
- Organization of the AWS file structure below data: https://github.com/owhite/AWS_NEMO/blob/main/NEMO_organization.md
- The file formats used in the sequence directories: https://github.com/nemoarchive/documentation/blob/master/file_extensions.md

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):

import json
import anndata as ad
import tempfile

from pprint import pprint

import boto3, polars, matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import seaborn as sns

Next, we will review the structure of dataset by listing the top level directories in our S3 bucket:

In [ ]:
### Review entry of the NeMO S3 bucket

# Top-level NeMO S3 bucket 
bucket = "nemo-test-filesystem"

chandelier_key = "data/biccn/lein/transcriptome/sncell/10x_v3/human/processed/counts/human_var_scVI_Chandelier.h5ad"
endo_key = "data/biccn/lein/transcriptome/sncell/10x_v3/human/processed/counts/human_var_scVI_Endo.h5ad"


# This is a public bucket and we don't need to sign requests.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])


At the top-level dataset, we see that the data have been separated into three programs: "bican", "biccn", and "other"

In [ ]:
# List the key prefixes within the top level 'data' prefix
for item in s3.list_objects_v2(Bucket=bucket, Prefix='data/', Delimiter='/', MaxKeys=10)['CommonPrefixes']:
    print(item['Prefix'])


Next, let's introduce some terms:

**PROGRAM:** A program is a coordinated, multi-institution research effort funded under a common strategic initiative (typically by NIH or another funding agency) to address a broad scientific objective. A program encompasses multiple independently led projects that collectively contribute to shared goals, standards, and deliverables. Programs often span multiple institutions and laboratories, and include multiple principal investigators (PIs).

**PROJECT:** A project is a defined research effort within a program, typically led by a principal investigator or investigator team, focused on a specific scientific question, dataset, or technical objective. Projects operate semi-independently but align with the broader goals and standards of the program.

There are two important caveats to the project definition: 
- For legacy reasons "grant" and "project" are used interchangeably. 
- Multiple funded efforts can be agregated into one project. 

The intention is that our heirarchical directory tree reflects this aggregation:
<pre>
Program (e.g., bican)
├── Project A (PI-led)
├── Project B (PI-led)
├── Project C (multi-institution)
│   ├── Project C.1 (PI-led)
│   └── Project C.2 (PI-led)
└── Project D (PI-led)
</pre>

So as we showed before first tier of the data directory structure reflects the program:
<pre>
bican/
biccn/
other/
</pre>

Then what you see here is that "grant" is showing a collection of projects, with the term "rf1_macosko" reflecting the project name, as in: 

<pre>
biccn/
    grant/
        rf1_macosko/
</pre>

In some cases there are grant-projects nested under the top-level program, e.g.: 

<pre>
bican/
    grant/
        BICAN_Dev_Mouse/
            aibs/
        BICAN_Dev_Multiomics/
            ucla_luo/
</pre>

Further down the heirarchical, several sequencing modalities are used for these projects, such as transcriptome, epigenome, multimodal and are reflected in the naming structure of next tier of the directory tree. Under each of these trees we have listed directories that contain the sequence information, e.g.: 

<pre>
bican/
    grant/
        BICAN_Dev_Mouse/
            aibs/
                transcriptome/
                    cells/
                        SEQUENCE_DIRECTORIES:296
</pre>

This document provides the entire NeMO human and mammalian brain atlas with the truncated sequence directories: [nemo_tree_pruned.txt](nemo_tree_pruned.txt)

We can get a partial listing of all our directories inside of /data/other with the program below.

In [ ]:
# List the directories within the 'data/other' prefix.
for item in s3.list_objects_v2(Bucket=bucket, Prefix='data/other/', MaxKeys=100)['Contents']:
    print(item['Key'])

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Not all, but the bulk of our data are H5AD files.  H5AD is common data format used to store single-cell biology datasets (especially RNA sequencing). They are widely used in computational biology workflows such as scanpy. A .h5ad file stores a matrix of measurements plus rich metadata in a single binary container which often includes:
- rows = cells
- columns = genes
- values = gene expression counts
- metadata = annotations about cells, genes, experiments

Think of it as a spreadsheet that can contain metadata tables, embeddings and graphs all stored efficiently in one file. 

A more complete list of files that can be obtained from the NeMO Archive is listed here: [[LINK](https://github.com/nemoarchive/documentation/blob/master/file_extensions.md)].

### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, let us load up and look into some package data as found in https://nemo-test-filesystem.s3.us-east-2.amazonaws.com/data/biccn/lein/transcriptome/sncell/10x_v3/human/processed/counts/human_var_scVI_Endo.h5ad

In [ ]:
# Open a file and run anndata on it

file_key = "data/biccn/lein/transcriptome/sncell/10x_v3/human/processed/counts/human_var_scVI_Endo.h5ad"

with tempfile.NamedTemporaryFile(suffix=".h5ad") as tmp_file:
    s3.download_file(bucket, file_key, tmp_file.name)
    adata = ad.read_h5ad(tmp_file.name)

print(adata)
print("\nobs columns:")
print(list(adata.obs.columns))
print("\nvar columns:")
print(list(adata.var.columns))
print("\nobs preview:")
print(adata.obs.head(10))
print("\nvar preview:")
print(adata.var.head(10))

In this example, we download a single `.h5ad` file from the NeMO S3 bucket and open it with `anndata`, a Python package commonly used for working with single-cell genomics data. Rather than treating the file as raw binary data, `anndata` interprets it as a structured dataset containing a data matrix together with annotations about cells and genes.

We then print a summary of the `AnnData` object and preview the `obs` and `var` tables. This gives us a quick view of the dataset’s dimensions, the metadata available for cells and features, and the general structure of the file before moving on to more detailed exploration.

In [ ]:
print("Shape:", adata.shape)
print("\nobs columns:")
print(list(adata.obs.columns))
print("\nvar columns:")
print(list(adata.var.columns))

print("\nFirst 5 rows of obs:")
display(adata.obs.head())

print("\nFirst 5 rows of var:")
display(adata.var.head())

### Q: A picture is worth a thousand words. Show us a visual from your dataset that either illustrates something informative about your dataset.

In this example, we use the `AnnData` object to calculate the total signal recorded for each cell and then plot those values as a histogram. This gives us a quick view of how counts are distributed across the dataset and helps reveal whether most cells have similar coverage or whether there is a wide spread in sequencing depth.

Visual summaries like this are often a useful first quality-control step in single-cell analysis. They can help identify unusually low-coverage or high-coverage cells and provide a first impression of the overall structure of the dataset before moving on to downstream analysis.


In [ ]:
cell_totals = np.asarray(adata.X.sum(axis=1)).ravel()

plt.hist(cell_totals, bins=50)
plt.xlabel("Total counts per cell")
plt.ylabel("Number of cells")
plt.title("Distribution of total counts per cell")
plt.show()

In [ ]:
# Violin plots from the current AnnData object

# First, reproduce the metadata-style violin plots from adata.obs.
obs_candidates = ["total_genes_detected", "total_counts", "total.reads"]
obs_cols = [col for col in obs_candidates if col in adata.obs.columns]

if obs_cols:
    fig, axes = plt.subplots(1, len(obs_cols), figsize=(5 * len(obs_cols), 5))

    if len(obs_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, obs_cols):
        plot_df = adata.obs[[col]].dropna().copy()
        sns.violinplot(data=plot_df, y=col, ax=ax, inner="quart")
        sns.stripplot(data=plot_df, y=col, ax=ax, color="black", size=1, alpha=0.15)
        ax.set_title(col)

    plt.tight_layout()
    plt.show()
else:
    print("None of the expected metadata columns were found in adata.obs.")
    print("Available obs columns:")
    print(list(adata.obs.columns))


# Next, reproduce the gene-expression violin plots.
gene_candidates = ["CYTB", "GFAP", "TUBB2B", "SYN2"]
genes = [gene for gene in gene_candidates if gene in adata.var_names]

if genes:
    expr = adata[:, genes].X
    if hasattr(expr, "toarray"):
        expr = expr.toarray()

    expr_df = pd.DataFrame(expr, columns=genes)
    long_df = expr_df.melt(var_name="gene", value_name="expression")

    plt.figure(figsize=(8, 5))
    sns.violinplot(data=long_df, x="gene", y="expression", inner="quart")
    plt.title("Expression distributions for selected genes")
    plt.tight_layout()
    plt.show()
else:
    print("None of the expected genes were found in adata.var_names.")
    print("Example available genes:")
    print(list(adata.var_names[:20]))


In this example, we use the `AnnData` object to calculate the total signal recorded for each cell and then plot those values as a histogram. This gives us a quick view of how counts are distributed across the dataset and helps reveal whether most cells have similar coverage or whether there is a wide spread in sequencing depth.

Visual summaries like this are often a useful first quality-control step in single-cell analysis. They can help identify unusually low-coverage or high-coverage cells and provide a first impression of the overall structure of the dataset before moving on to downstream analysis.


<!-- DATA PROVIDER INSTRUCTIONS
This section is less prescriptive / freeform than previous sections. The goal here is to show an opinionated example of answering a question using your data. The scale of your dataset may preclude a full example, and so feel free to limit the scope of this example (e.g. work on a subset of data). Users should be able to replicate your example in this notebook, and get a sense of how they would scale up.

A "toy" example is better than no example.

Ideally, your example would:
1. Transmit some of your domain & dataset experience to the reader, drawing on your own work as much as possible
2. Provide a jumping off point for users to extend your work, and do novel work of their own.

DATA PROVIDER INSTRUCTIONS -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

EXAMPLE - REPLACE

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Praesent sem enim, finibus vel leo vel, blandit interdum tortor. In lectus dolor, congue eu odio vel, euismod facilisis lacus. Vestibulum aliquet, quam in rhoncus tempus, arcu massa suscipit lacus, nec fermentum quam justo nec lacus. Duis id leo fermentum ante tempor pulvinar eu vitae ligula. Cras feugiat vel ligula sit amet lacinia. Mauris sit amet sem vestibulum ligula volutpat iaculis in eu velit. Sed turpis magna, porta ac nisi vitae, maximus volutpat mi. Vestibulum mattis est eros, nec pellentesque nisi iaculis sed. 

EXAMPLE - REPLACE

<!-- DATA PROVIDER INSTRUCTIONS
This section is, like the previous one, intended to be freeform / non-prescriptive. The goal here is to provide a challenge to the community to do something novel with your dataset. That can either be novel in terms of the task, or novel in terms of methodological or computational approach.

Another way to consider this section, is as a wishlist. If you were less constrained by time, cost, skill, etc., what would you like to see achieved using these data? 

The challenge should, however, be somewhat realistic. A challenge that assumes e.g. original data collection, is likely to go unanswered.
DATA PROVIDER INSTRUCTIONS -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

EXAMPLE - REPLACE

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Praesent sem enim, finibus vel leo vel, blandit interdum tortor. In lectus dolor, congue eu odio vel, euismod facilisis lacus. Vestibulum aliquet, quam in rhoncus tempus, arcu massa suscipit lacus, nec fermentum quam justo nec lacus. Duis id leo fermentum ante tempor pulvinar eu vitae ligula. Cras feugiat vel ligula sit amet lacinia. Mauris sit amet sem vestibulum ligula volutpat iaculis in eu velit. Sed turpis magna, porta ac nisi vitae, maximus volutpat mi. Vestibulum mattis est eros, nec pellentesque nisi iaculis sed. 

EXAMPLE - REPLACE